<a href="https://colab.research.google.com/github/AlwayszZZZ/4131Downloads0302/blob/main/0428Lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
apt update
apt install -y python3-pip python3-venv git

In [ ]:
python3 -m venv /root/llm-env
source /root/llm-env/bin/activate
pip install -U pip

In [ ]:
pip install huggingface_hub
pip install vllm

In [ ]:
huggingface-cli login

In [ ]:
## 我的，数据库下载
# from datasets import load_dataset

# ds = load_dataset("Hieu-Pham/kaggle_food_recipes")

In [ ]:
#这个是自己安装模型的路径（？）
pip install -U "huggingface-hub>=0.34.0,<1.0"
huggingface-cli download Qwen/Qwen2.5-7B-Instruct \
  --local-dir /root/models/Qwen2.5-7B-Instruct

python -m vllm.entrypoints.openai.api_server \
  --model /root/models/Qwen2.5-7B-Instruct \.
  --served-model-name qwen \
  --host 0.0.0.0 \
  --port 8000 \
  --api-key test-key

In [ ]:
python -m pip install datasets transformers peft trl accelerate pandas bitsandbytes

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
)
from peft import LoraConfig
from trl import SFTTrainer

MODEL_NAME = "/root/models/Qwen2.5-7B-Instruct"
DATA_PATH = "data/food_recipes.jsonl"
OUTPUT_DIR = "outputs/qwen25-7b-food-recipes-lora"

MAX_LENGTH = 2048

# 1. tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 2. dataset
dataset = load_dataset("json", data_files=DATA_PATH, split="train")
dataset = dataset.train_test_split(test_size=0.02, seed=42)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

def formatting_func(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return text

# 3. model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True,
)

model.config.use_cache = False

# 4. LoRA
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

# 5. training args
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=100,
    save_total_limit=2,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    bf16=torch.cuda.is_available(),
    fp16=False,
    gradient_checkpointing=True,
    report_to="none",
    optim="adamw_torch",
)
# 6. trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    formatting_func=formatting_func,
    max_length=MAX_LENGTH,
)

trainer.train()

# 保存LoRA权重
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"✅ 微调完成！保存到 {OUTPUT_DIR}")

In [ ]:
# 跑完后输出一个
# /path/to/lora_adapter
# 训练结束后，把：
# -  base model
# -  LoRA adapter
# 合并成一个新模型